In [ ]:
# ===============================
# 0. IMPORTS
# ===============================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs
import numpy as np

from transformers import BertTokenizer, BertModel

# ===============================
# 1. DEVICE
# ===============================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ===============================
# 2. SMILES → GRAPH
# ===============================
def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    x = []
    for atom in mol.GetAtoms():
        x.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            atom.GetFormalCharge(),
            int(atom.GetHybridization())
        ])
    x = torch.tensor(x, dtype=torch.float)

    edge_index = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])

    if len(edge_index) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    return Data(x=x, edge_index=edge_index)

# ===============================
# 3. DOMAIN FEATURES
# ===============================
def morgan_fp(smiles, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)
    arr = np.zeros((n_bits,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return torch.tensor(arr, dtype=torch.float)

AA = 'ACDEFGHIKLMNPQRSTVWY'

def protein_aac(seq):
    seq = seq.upper()
    if len(seq) == 0:
        return torch.zeros((len(AA),), dtype=torch.float)
    return torch.tensor([seq.count(a)/len(seq) for a in AA], dtype=torch.float)

# ===============================
# 4. DATASET
# ===============================
class CPIDataset(Dataset):
    def __init__(self, file_path):
        self.samples = []
        with open(file_path) as f:
            for line in f:
                s, p, y = line.strip().split()
                self.samples.append((s, p, int(y)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        smiles, protein, label = self.samples[idx]

        data = smiles_to_graph(smiles)
        data.fp = morgan_fp(smiles).unsqueeze(0)
        data.aac = protein_aac(protein).unsqueeze(0)
        data.protein_seq = protein
        data.y = torch.tensor(label, dtype=torch.float)

        return data

# ===============================
# 5. MODEL COMPONENTS
# ===============================
class CompoundGNN(nn.Module):
    def __init__(self, node_dim=4, hidden=128):
        super().__init__()
        self.conv1 = GCNConv(node_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return global_mean_pool(x, batch)

class ProteinEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokenizer = BertTokenizer.from_pretrained(
            "Rostlab/prot_bert", do_lower_case=False
        )
        self.model = BertModel.from_pretrained("Rostlab/prot_bert")

        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, seqs):
        seqs = [" ".join(list(s)) for s in seqs]
        inputs = self.tokenizer(
            seqs,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        out = self.model(**inputs)
        return out.last_hidden_state[:, 0, :]  # CLS

class HybridCPI(nn.Module):
    def __init__(self):
        super().__init__()
        self.gnn = CompoundGNN()
        self.protein = ProteinEncoder()

        fusion_dim = 128 + 2048 + 1024 + 20

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, batch):
        gnn_emb = self.gnn(batch.x, batch.edge_index, batch.batch)
        # after batching, fp and aac will have shape (batch_size, dim)
        fp = batch.fp
        aac = batch.aac
        prot_emb = self.protein(batch.protein_seq)

        return self.classifier(torch.cat([gnn_emb, fp, prot_emb, aac], dim=1)).squeeze()

# ===============================
# 6. LOSS
# ===============================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

# ===============================
# 7. TRAIN LOOP
# ===============================
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        preds = model(batch)
        loss = loss_fn(preds, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# ===============================
# 8. MAIN
# ===============================
if __name__ == "__main__":

    data_path = "sample_data.txt"
    data_path = "../dataset/b_cancer/original/data.txt"

    dataset = CPIDataset(data_path)
    loader = DataLoader(dataset, batch_size=2, shuffle=True)

    model = HybridCPI().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    loss_fn = FocalLoss()

    for epoch in range(5):
        loss = train_epoch(model, loader, optimizer, loss_fn)
        print(f"Epoch {epoch+1} | Loss: {loss:.4f}")


c:\Users\Dell\miniconda3\envs\cpi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


[10:30:59] DEPRECATION WARNING: please use MorganGenerator
[10:30:59] DEPRECATION WARNING: please use MorganGenerator
[10:31:06] DEPRECATION WARNING: please use MorganGenerator
[10:31:06] DEPRECATION WARNING: please use MorganGenerator
[10:31:14] DEPRECATION WARNING: please use MorganGenerator
[10:31:14] DEPRECATION WARNING: please use MorganGenerator
[10:31:21] DEPRECATION WARNING: please use MorganGenerator
[10:31:21] DEPRECATION WARNING: please use MorganGenerator
[10:31:28] DEPRECATION WARNING: please use MorganGenerator
[10:31:28] DEPRECATION WARNING: please use MorganGenerator
[10:31:35] DEPRECATION WARNING: please use MorganGenerator
[10:31:35] DEPRECATION WARNING: please use MorganGenerator
[10:31:42] DEPRECATION WARNING: please use MorganGenerator
[10:31:42] DEPRECATION WARNING: please use MorganGenerator
[10:31:49] DEPRECATION WARNING: please use MorganGenerator
[10:31:49] DEPRECATION WARNING: please use MorganGenerator
[10:31:56] DEPRECATION WARNING: please use MorganGenerat

In [5]:
# ===============================
# 9. EVALUATION
# ===============================
# Computes Accuracy, Precision, Recall and AUC on a dataset.
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
import os

# try sample path used above, fallback to original dataset if available
data_path = "sample_data.txt"
if not os.path.exists(data_path) and os.path.exists("../dataset/b_cancer/original/data.txt"):
    data_path = "../dataset/b_cancer/original/data.txt"

print('Using evaluation data:', data_path)

eval_dataset = CPIDataset(data_path)
from torch_geometric.loader import DataLoader as PyGDataLoader
loader = PyGDataLoader(eval_dataset, batch_size=32, shuffle=False)

model = HybridCPI().to(device)
ckpt_path = "model.pt"
if os.path.exists(ckpt_path):
    print('Loading checkpoint:', ckpt_path)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
else:
    print('No checkpoint found at', ckpt_path, '- using current model weights')

model.eval()
all_probs = []
all_preds = []
all_targets = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)
        logits = model(batch)
        probs = torch.sigmoid(logits).cpu().numpy()
        y = batch.y.cpu().numpy()

        probs = probs.reshape(-1)
        preds = (probs >= 0.5).astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_targets.extend(y.reshape(-1).tolist())

import numpy as np
all_probs = np.array(all_probs)
all_preds = np.array(all_preds)
all_targets = np.array(all_targets)

acc = accuracy_score(all_targets, all_preds)
prec = precision_score(all_targets, all_preds, zero_division=0)
rec = recall_score(all_targets, all_preds, zero_division=0)
try:
    auc = roc_auc_score(all_targets, all_probs)
except Exception as e:
    auc = None
    print('AUC could not be computed:', e)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall: {rec:.4f}")
print(f"AUC: {auc if auc is not None else 'N/A'}")


Using evaluation data: sample_data.txt
No checkpoint found at model.pt - using current model weights


[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator
[15:44:56] DEPRECATION WARNING: please use MorganGenerator


Accuracy: 0.2500
Precision: 0.0000
Recall: 0.0000
AUC: 0.41666666666666663
